In [ ]:
from torch.utils.data import Dataset, DataLoader, Subset
import rasterio
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import os, random, warnings
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
print("Libraries loaded.")


In [ ]:
IMAGE_DIR = "/kaggle/input/datasets/petermoawad/satalite/data-20260926T092652Z-1-001/data/images"
LABEL_DIR = "/kaggle/input/datasets/petermoawad/satalite/data-20260926T092652Z-1-001/data/labels"

image_files = sorted([f for f in os.listdir(IMAGE_DIR) if f.endswith(".tif")])
label_files = sorted([f for f in os.listdir(LABEL_DIR) if f.endswith(".png")])

# ── Verify actual band metadata from the TIF file ────────────────────────────
sample_path = os.path.join(IMAGE_DIR, image_files[0])
with rasterio.open(sample_path) as src:
    n_bands, h, w = src.count, src.height, src.width
    band_descs     = src.descriptions          # may contain band names
    sample_img     = src.read().astype(np.float32)

print("=" * 60)
print("  DATASET STRUCTURE")
print("=" * 60)
print(f"  Total samples : {len(image_files)}")
print(f"  Image shape   : ({n_bands}, {h}, {w})  [bands x H x W]")
print(f"  Label format  : PNG, binary (0=background, 255=water)")
print()
print("  Band metadata from TIF file:")
for i, d in enumerate(band_descs):
    mean_val = sample_img[i].mean()
    print(f"    ch{i:2d}: description={d!r:25s}  sample_mean={mean_val:.1f}")

# ── Class distribution (from all labels — for information only) ───────────────
water_px = total_px = 0
for fname in label_files:
    lbl = np.array(Image.open(os.path.join(LABEL_DIR, fname)))
    water_px += (lbl > 0).sum()
    total_px += lbl.size

water_pct = 100 * water_px / total_px
print()
print(f"  Water pixels  : {water_px:,}  ({water_pct:.1f}%)")
print(f"  Background    : {total_px - water_px:,}  ({100 - water_pct:.1f}%)")
print(f"  Class imbalance ratio : {(100-water_pct)/water_pct:.1f}x")


In [ ]:
# ── Band Descriptions (Sentinel-2 MSI 12-band ordering assumed) ──────────────
# NOTE: actual band identity is verified above from rasterio metadata.
# Adjust GREEN_CH / NIR_CH / SWIR_CH if your dataset uses a different ordering.

BAND_INFO = [
    (0,  "B1  Coastal/Aerosol", "~443 nm",
     "Aerosol detection, shallow water penetration. "
     "Water is moderately reflective — used for clear/shallow water mapping."),
    (1,  "B2  Blue",            "~490 nm",
     "Deep water penetration. "
     "Used in true-colour RGB. Turbid water is brighter than clear water."),
    (2,  "B3  Green",           "~560 nm",
     "Chlorophyll reflection peak. KEY INPUT for NDWI & MNDWI. "
     "Water has moderate reflectance here."),
    (3,  "B4  Red",             "~665 nm",
     "Strong chlorophyll absorption. "
     "Water appears very dark — high contrast with dry soil."),
    (4,  "B5  Red-Edge",        "~705 nm",
     "Vegetation red-edge. Water absorbs almost completely."),
    (5,  "B6  Red-Edge",        "~740 nm",
     "Vegetation structure. Helps separate flooded vegetation from open water."),
    (6,  "B7  Red-Edge",        "~783 nm",
     "Near-IR transition. Water absorbs strongly."),
    (7,  "B8  NIR (broad)",     "~842 nm",
     "Water absorbs almost completely -> appears black. "
     "Core band for NDWI = (Green - NIR)/(Green + NIR)."),
    (8,  "B8A Narrow NIR",      "~865 nm",
     "Narrower NIR. Very high water absorption."),
    (9,  "B9  Water Vapour",    "~945 nm",
     "Atmospheric water-vapour; mainly used for atmospheric correction."),
    (10, "B11 SWIR-1",          "~1610 nm",
     "Water absorbs completely -> very dark. "
     "Core band for MNDWI = (Green - SWIR)/(Green + SWIR)."),
    (11, "B12 SWIR-2",          "~2190 nm",
     "Soil moisture & water content. Water absorbs completely."),
]

# ── Choose correct band indices for water indices ─────────────────────────────
# Based on Sentinel-2 B03 / B08 / B11 ordering (verified by metadata above).
# Change these if band_descs above shows a different order.
GREEN_CH = 2    # B3  Green  ~560 nm
NIR_CH   = 7    # B8  NIR    ~842 nm
SWIR_CH  = 10   # B11 SWIR-1 ~1610 nm

print("Band table:")
print(f"  {'ch':>3}  {'Name':20s}  {'Wavelength':10s}  Water effect")
print("  " + "-" * 70)
for ch, name, wl, desc in BAND_INFO:
    effect = desc.split(".")[0]
    marker = " <-- GREEN" if ch == GREEN_CH else " <-- NIR" if ch == NIR_CH else " <-- SWIR" if ch == SWIR_CH else ""
    print(f"  {ch:3d}  {name:20s}  {wl:10s}  {effect}{marker}")

# ── Validate index choices ────────────────────────────────────────────────────
# NDWI is meaningful only when Green reflectance > NIR reflectance for water
sample_ndwi = (sample_img[GREEN_CH] - sample_img[NIR_CH]) / (sample_img[GREEN_CH] + sample_img[NIR_CH] + 1e-8)
print(f"\n  Sample NDWI  stats: mean={sample_ndwi.mean():.3f}  "
      f"(positive = water detected, expected for water-rich scenes)")
sample_mndwi = (sample_img[GREEN_CH] - sample_img[SWIR_CH]) / (sample_img[GREEN_CH] + sample_img[SWIR_CH] + 1e-8)
print(f"  Sample MNDWI stats: mean={sample_mndwi.mean():.3f}")


In [ ]:
def norm_display(band, lo=2, hi=98):
    lo_v, hi_v = np.percentile(band, lo), np.percentile(band, hi)
    return np.clip((band - lo_v) / (hi_v - lo_v + 1e-8), 0, 1)

mid = len(image_files) // 2
with rasterio.open(os.path.join(IMAGE_DIR, image_files[mid])) as sr:
    viz_img = sr.read().astype(np.float32)
viz_lbl = (np.array(Image.open(os.path.join(LABEL_DIR, label_files[mid]))) > 0).astype(np.float32)

# ── (A) All 12 spectral bands ──────────────────────────────────────────────
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle("All 12 Spectral Bands — Single Sample\n"
             "(water appears dark in NIR/SWIR; brighter in visible bands)",
             fontsize=13, fontweight="bold")
for c, ax in enumerate(axes.flat):
    ch, name, wl, _ = BAND_INFO[c]
    ax.imshow(norm_display(viz_img[c]), cmap="gray")
    ax.set_title(f"ch{c}: {name}\n{wl}", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.savefig("bands_12.png", dpi=150, bbox_inches="tight")
plt.show()

# ── (B) Composites + Water indices + Mask ──────────────────────────────────
ndwi_v  = (viz_img[GREEN_CH] - viz_img[NIR_CH])  / (viz_img[GREEN_CH] + viz_img[NIR_CH]  + 1e-8)
mndwi_v = (viz_img[GREEN_CH] - viz_img[SWIR_CH]) / (viz_img[GREEN_CH] + viz_img[SWIR_CH] + 1e-8)

rgb = np.stack([norm_display(viz_img[3]), norm_display(viz_img[2]), norm_display(viz_img[1])], -1)
fc  = np.stack([norm_display(viz_img[7]), norm_display(viz_img[3]), norm_display(viz_img[2])], -1)

ndwi_pred = (ndwi_v > 0).astype(np.float32)
overlap   = np.zeros((*ndwi_pred.shape, 3))
overlap[..., 0] = (ndwi_pred == 1) & (viz_lbl == 0)
overlap[..., 1] = (ndwi_pred == 1) & (viz_lbl == 1)
overlap[..., 2] = (ndwi_pred == 0) & (viz_lbl == 1)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("RGB Composites, Water Indices & Ground-Truth Mask", fontsize=13, fontweight="bold")

axes[0,0].imshow(rgb);         axes[0,0].set_title("True-Colour RGB (B4/B3/B2)"); axes[0,0].axis("off")
axes[0,1].imshow(fc);          axes[0,1].set_title("False-Colour NIR (B8/B4/B3)\nWater=dark"); axes[0,1].axis("off")
axes[0,2].imshow(viz_lbl, cmap="Blues")
axes[0,2].set_title(f"Ground-Truth Mask\nWater={viz_lbl.mean()*100:.1f}%"); axes[0,2].axis("off")

im1 = axes[1,0].imshow(ndwi_v,  cmap="RdBu", vmin=-1, vmax=1)
axes[1,0].set_title("NDWI = (Green - NIR) / (Green + NIR)\nPositive = water"); axes[1,0].axis("off")
plt.colorbar(im1, ax=axes[1,0], fraction=0.046)

im2 = axes[1,1].imshow(mndwi_v, cmap="RdBu", vmin=-1, vmax=1)
axes[1,1].set_title("MNDWI = (Green - SWIR) / (Green + SWIR)\nPositive = water"); axes[1,1].axis("off")
plt.colorbar(im2, ax=axes[1,1], fraction=0.046)

axes[1,2].imshow(overlap)
axes[1,2].set_title("NDWI>0 vs Mask\nGreen=TP  Red=FP  Blue=FN"); axes[1,2].axis("off")

plt.tight_layout()
plt.savefig("composites_indices.png", dpi=150, bbox_inches="tight")
plt.show()
print("Visualisations saved.")


## Research: Multispectral Segmentation & U-Net Architecture

### Multispectral vs RGB

| | RGB | Multispectral (12-ch) |
|--|--|--|
| Channels | 3 | 12 |
| Wavelength | 400–700 nm | 443–2190 nm |
| Water signal | Weak | Strong (NIR/SWIR absorbed by water) |

**Why NIR/SWIR are key for water:**  
Water strongly absorbs near-infrared (NIR ~842 nm) and shortwave infrared (SWIR ~1610 nm), making water bodies appear nearly **black** — a strong discriminative signal absent from RGB.

**Feature Engineering — Water Indices:**
- **NDWI** = (Green − NIR) / (Green + NIR) → positive for water, negative for land
- **MNDWI** = (Green − SWIR) / (Green + SWIR) → better at separating water from built-up areas

Adding NDWI + MNDWI as two extra channels gives the model **pre-computed physics-based water features**.

---

### U-Net Architecture

```
Input (14, 128, 128)
    ↓ Encoder (×4 levels) — downsample, double channels
    ↓ Bottleneck
    ↑ Decoder (×4 levels) — upsample + skip connections from encoder
    ↓
Output (1, 128, 128)  →  sigmoid  →  binary water mask
```

**Residual U-Net (Final Model):** Adds residual connections within each encoder/decoder block.  
Each block: `output = ReLU(Conv-BN-ReLU-Conv-BN(x) + shortcut(x))`  
Benefits: better gradient flow, faster convergence, improved segmentation accuracy.

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    def forward(self, logits, targets):
        p = torch.sigmoid(logits)
        inter = (p * targets).sum(dim=(2, 3))
        union = p.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
        return 1 - ((2 * inter + self.smooth) / (union + self.smooth)).mean()

class CombinedLoss(nn.Module):
    def __init__(self, pos_weight=None, bce_w=0.5, dice_w=0.5):
        super().__init__()
        self.bce    = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        self.dice   = DiceLoss()
        self.bce_w  = bce_w
        self.dice_w = dice_w
    def forward(self, logits, targets):
        return self.bce_w * self.bce(logits, targets) + self.dice_w * self.dice(logits, targets)

print("DiceLoss, CombinedLoss defined.")


In [ ]:
class WaterDataset(Dataset):
    """Supports 12-channel (baseline) or 14-channel (+ NDWI/MNDWI) modes."""
    def __init__(self, image_dir, label_dir, means, stds, augment=False, use_indices=False):
        self.image_dir   = image_dir
        self.label_dir   = label_dir
        self.means       = means          # shape (12,) or (14,)
        self.stds        = stds           # shape (12,) or (14,)
        self.augment     = augment
        self.use_indices = use_indices    # True -> 14-ch with NDWI/MNDWI
        self.n_ch        = 14 if use_indices else 12

        img_names   = set(os.path.splitext(f)[0] for f in os.listdir(image_dir))
        lbl_names   = set(os.path.splitext(f)[0] for f in os.listdir(label_dir))
        matched     = sorted(img_names & lbl_names)
        self.image_files = [f"{n}.tif" for n in matched]
        self.label_files = [f"{n}.png" for n in matched]

    def _add_indices(self, img):
        g, nir, swir = img[GREEN_CH], img[NIR_CH], img[SWIR_CH]
        ndwi  = (g - nir)  / (g + nir  + 1e-8)
        mndwi = (g - swir) / (g + swir + 1e-8)
        return np.concatenate([img, ndwi[None], mndwi[None]], axis=0)

    def _augment(self, img, lbl):
        if random.random() > 0.5:
            img = np.flip(img, axis=2).copy(); lbl = np.flip(lbl, axis=2).copy()
        if random.random() > 0.5:
            img = np.flip(img, axis=1).copy(); lbl = np.flip(lbl, axis=1).copy()
        k = random.randint(0, 3)
        img = np.rot90(img, k=k, axes=(1, 2)).copy()
        lbl = np.rot90(lbl, k=k, axes=(1, 2)).copy()
        return img, lbl

    def __len__(self): return len(self.image_files)

    def __getitem__(self, idx):
        with rasterio.open(os.path.join(self.image_dir, self.image_files[idx])) as sr:
            img = sr.read().astype(np.float32)          # (12, H, W)

        if self.use_indices:
            img = self._add_indices(img)                # (14, H, W)

        # Dataset-wise per-channel normalisation (stats from training set only)
        for c in range(self.n_ch):
            img[c] = (img[c] - self.means[c]) / (self.stds[c] + 1e-8)

        lbl = np.array(Image.open(os.path.join(self.label_dir, self.label_files[idx]))).astype(np.float32)
        lbl = (lbl > 0).astype(np.float32)[None]       # (1, H, W)

        if self.augment:
            img, lbl = self._augment(img, lbl)

        return torch.tensor(img), torch.tensor(lbl)

print("WaterDataset defined (12-ch and 14-ch modes).")


In [ ]:
BATCH_SIZE = 16

# ── Step 1: Reproducible split FIRST (before any statistics) ─────────────────
n          = len(image_files)
train_size = int(0.8 * n)
val_size   = int(0.1 * n)
test_size  = n - train_size - val_size

generator = torch.Generator().manual_seed(42)
indices   = torch.randperm(n, generator=generator).tolist()
train_idx = indices[:train_size]
val_idx   = indices[train_size : train_size + val_size]
test_idx  = indices[train_size + val_size :]

print(f"Split  train={train_size}  val={val_size}  test={test_size}")

# ── Step 2: Compute normalisation stats from TRAINING data only ──────────────
# Using only training files prevents data leakage into val/test.
train_files = [image_files[i] for i in train_idx]

ch_px = [[] for _ in range(12)]
ndwi_px, mndwi_px = [], []

for fname in train_files:
    with rasterio.open(os.path.join(IMAGE_DIR, fname)) as sr:
        img = sr.read().astype(np.float32)
    for c in range(12):
        ch_px[c].append(img[c].flatten())
    g, nir, swir = img[GREEN_CH], img[NIR_CH], img[SWIR_CH]
    ndwi_px.append( ((g - nir)  / (g + nir  + 1e-8)).flatten() )
    mndwi_px.append(((g - swir) / (g + swir + 1e-8)).flatten() )

means12 = np.array([np.concatenate(ch_px[c]).mean() for c in range(12)])
stds12  = np.array([np.concatenate(ch_px[c]).std()  for c in range(12)])
ndwi_all  = np.concatenate(ndwi_px)
mndwi_all = np.concatenate(mndwi_px)
means14 = np.append(means12, [ndwi_all.mean(), mndwi_all.mean()])
stds14  = np.append(stds12,  [ndwi_all.std(),  mndwi_all.std()])

print("Training-set-only statistics computed.")
print(f"  12-ch: means in [{means12.min():.1f}, {means12.max():.1f}]")
print(f"  NDWI  mean={means14[12]:.4f}  MNDWI mean={means14[13]:.4f}")

# ── Step 3: pos_weight from training labels only ──────────────────────────────
water_px = total_px = 0
for i in train_idx:
    lbl = np.array(Image.open(os.path.join(LABEL_DIR, label_files[i])))
    water_px += (lbl > 0).sum()
    total_px += lbl.size
pos_weight = torch.tensor([(total_px - water_px) / (water_px + 1e-8)])
print(f"  pos_weight = {pos_weight.item():.2f}  (training-set class ratio)")

# ── Step 4: Create Dataset and DataLoader objects ─────────────────────────────
# Separate augment=True/False objects — same stats applied to all splits.
def make_loaders(means, stds, use_indices, batch=BATCH_SIZE):
    train_ds = WaterDataset(IMAGE_DIR, LABEL_DIR, means, stds, augment=True,  use_indices=use_indices)
    eval_ds  = WaterDataset(IMAGE_DIR, LABEL_DIR, means, stds, augment=False, use_indices=use_indices)
    tr = DataLoader(Subset(train_ds, train_idx), batch_size=batch, shuffle=True,  num_workers=2, pin_memory=True)
    vl = DataLoader(Subset(eval_ds,  val_idx),   batch_size=batch, shuffle=False, num_workers=2, pin_memory=True)
    te = DataLoader(Subset(eval_ds,  test_idx),  batch_size=batch, shuffle=False, num_workers=2, pin_memory=True)
    return tr, vl, te

train_loader_12, val_loader_12, test_loader_12 = make_loaders(means12, stds12, use_indices=False)
train_loader_14, val_loader_14, test_loader_14 = make_loaders(means14, stds14, use_indices=True)

print(f"12-ch loaders: train={len(train_loader_12.dataset)}  val={len(val_loader_12.dataset)}  test={len(test_loader_12.dataset)}")
print(f"14-ch loaders: same sizes with NDWI/MNDWI channels added")


In [ ]:
# ── Baseline U-Net (12-channel, basic) ───────────────────────────────────────
class UNet(nn.Module):
    """Standard U-Net with 4 encoder-decoder levels.
    Baseline: input_channels=12.  Feature-Engineering run: input_channels=14."""
    def __init__(self, input_channels, output_channels=1):
        super().__init__()
        def conv_block(ic, oc):
            return nn.Sequential(
                nn.Conv2d(ic, oc, 3, padding=1, bias=False), nn.BatchNorm2d(oc), nn.ReLU(inplace=True),
                nn.Conv2d(oc, oc, 3, padding=1, bias=False), nn.BatchNorm2d(oc), nn.ReLU(inplace=True))

        self.enc1 = conv_block(input_channels, 64)
        self.enc2 = conv_block(64,  128)
        self.enc3 = conv_block(128, 256)
        self.enc4 = conv_block(256, 512)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = conv_block(512, 1024)
        self.up1  = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec1 = conv_block(1024, 512)
        self.up2  = nn.ConvTranspose2d(512,  256, 2, stride=2)
        self.dec2 = conv_block(512,  256)
        self.up3  = nn.ConvTranspose2d(256,  128, 2, stride=2)
        self.dec3 = conv_block(256,  128)
        self.up4  = nn.ConvTranspose2d(128,   64, 2, stride=2)
        self.dec4 = conv_block(128,   64)
        self.out  = nn.Conv2d(64, output_channels, 1)

    def forward(self, x):
        s1 = self.enc1(x);      x = self.pool(s1)
        s2 = self.enc2(x);      x = self.pool(s2)
        s3 = self.enc3(x);      x = self.pool(s3)
        s4 = self.enc4(x);      x = self.pool(s4)
        x  = self.bottleneck(x)
        x  = self.dec1(torch.cat([self.up1(x), s4], 1))
        x  = self.dec2(torch.cat([self.up2(x), s3], 1))
        x  = self.dec3(torch.cat([self.up3(x), s2], 1))
        x  = self.dec4(torch.cat([self.up4(x), s1], 1))
        return self.out(x)

print("UNet (Baseline) defined.")


In [ ]:
# ── Residual U-Net (14-channel, Final Model) ──────────────────────────────────
class ResidualBlock(nn.Module):
    """Conv block with residual (skip) connection within the block."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch))
        self.shortcut = (nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch))
                         if in_ch != out_ch else nn.Identity())
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.conv(x) + self.shortcut(x))

class ResidualUNet(nn.Module):
    """U-Net where every conv block is replaced by a ResidualBlock.
    Benefits: better gradient flow, faster convergence, ~same parameter count."""
    def __init__(self, input_channels, output_channels=1):
        super().__init__()
        self.enc1 = ResidualBlock(input_channels, 64)
        self.enc2 = ResidualBlock(64,  128)
        self.enc3 = ResidualBlock(128, 256)
        self.enc4 = ResidualBlock(256, 512)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ResidualBlock(512, 1024)
        self.up1  = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec1 = ResidualBlock(1024, 512)
        self.up2  = nn.ConvTranspose2d(512,  256, 2, stride=2)
        self.dec2 = ResidualBlock(512,  256)
        self.up3  = nn.ConvTranspose2d(256,  128, 2, stride=2)
        self.dec3 = ResidualBlock(256,  128)
        self.up4  = nn.ConvTranspose2d(128,   64, 2, stride=2)
        self.dec4 = ResidualBlock(128,   64)
        self.out  = nn.Conv2d(64, output_channels, 1)

    def forward(self, x):
        s1 = self.enc1(x);      x = self.pool(s1)
        s2 = self.enc2(x);      x = self.pool(s2)
        s3 = self.enc3(x);      x = self.pool(s3)
        s4 = self.enc4(x);      x = self.pool(s4)
        x  = self.bottleneck(x)
        x  = self.dec1(torch.cat([self.up1(x), s4], 1))
        x  = self.dec2(torch.cat([self.up2(x), s3], 1))
        x  = self.dec3(torch.cat([self.up3(x), s2], 1))
        x  = self.dec4(torch.cat([self.up4(x), s1], 1))
        return self.out(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_b = UNet(12, 1);          _r = ResidualUNet(14, 1)
print(f"UNet(12,1)       params: {sum(p.numel() for p in _b.parameters()):,}")
print(f"ResidualUNet(14,1) params: {sum(p.numel() for p in _r.parameters()):,}")
del _b, _r
print(f"Device: {device}")


In [ ]:
# Reusable training + evaluation functions 
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    preds_all, masks_all = [], []
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            out = model(imgs)
            total_loss += criterion(out, masks).item()
            preds_all.append((torch.sigmoid(out) > 0.5).cpu())
            masks_all.append(masks.cpu())
    preds = torch.cat(preds_all).numpy().flatten()
    gts   = torch.cat(masks_all).numpy().flatten()
    tp = (preds * gts).sum();  fp = (preds * (1-gts)).sum();  fn = ((1-preds) * gts).sum()
    union = preds.sum() + gts.sum() - tp
    iou   = tp / (union + 1e-8)
    prec  = tp / (tp + fp + 1e-8)
    rec   = tp / (tp + fn + 1e-8)
    f1    = 2 * prec * rec / (prec + rec + 1e-8)
    return {"loss": total_loss / len(loader), "iou": float(iou),
            "precision": float(prec), "recall": float(rec), "f1": float(f1)}

def train_model(model, train_loader, val_loader, name,
                epochs=100, lr=1e-4, patience=10):
    criterion = CombinedLoss(pos_weight=pos_weight.to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=5, min_lr=1e-6)

    history  = {"train_loss":[], "val_loss":[], "iou":[], "precision":[], "recall":[], "f1":[]}
    best_val = float("inf")
    no_improve = 0
    save_path  = f"{name}_best.pth"

    for epoch in range(epochs):
        # Train
        model.train()
        t_loss = 0
        for imgs, masks in train_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), masks)
            loss.backward(); optimizer.step()
            t_loss += loss.item()
        avg_train = t_loss / len(train_loader)

        # Validate
        m = evaluate(model, val_loader, criterion, device)
        scheduler.step(m["loss"])

        history["train_loss"].append(avg_train)
        history["val_loss"].append(m["loss"])
        history["iou"].append(m["iou"])
        history["precision"].append(m["precision"])
        history["recall"].append(m["recall"])
        history["f1"].append(m["f1"])

        if m["loss"] < best_val:
            best_val   = m["loss"]
            no_improve = 0
            torch.save(model.state_dict(), save_path)
        else:
            no_improve += 1

        lr_cur = optimizer.param_groups[0]["lr"]
        print(f"[{name}] E{epoch+1:03d}/{epochs} | lr={lr_cur:.1e} | "
              f"train={avg_train:.4f} val={m['loss']:.4f} | "
              f"IoU={m['iou']:.4f} F1={m['f1']:.4f}")

        if no_improve >= patience:
            print(f"  Early stopping at epoch {epoch+1}")
            break

    model.load_state_dict(torch.load(save_path, map_location=device))
    print(f"\n[{name}] Best val loss: {best_val:.4f}")
    return history, save_path

print("evaluate() and train_model() defined.")


In [ ]:
# BASELINE: 12-channel standard U-Net
print("=" * 60)
print("  BASELINE — UNet(12, 1)  |  12 spectral bands")
print("=" * 60)

baseline_model = UNet(12, 1).to(device)
history_baseline, _ = train_model(
    baseline_model, train_loader_12, val_loader_12,
    name="baseline", epochs=100, lr=1e-4, patience=10)


In [ ]:
# FEATURE ENGINEERING: 14-channel standard U-Net (12 bands + NDWI + MNDWI)
print("=" * 60)
print("  FEATURE ENG — UNet(14, 1)  |  12 bands + NDWI + MNDWI")
print("=" * 60)

fe_model = UNet(14, 1).to(device)
history_fe, _ = train_model(
    fe_model, train_loader_14, val_loader_14,
    name="fe_unet", epochs=100, lr=1e-4, patience=10)


In [ ]:
# FINAL MODEL: 14-channel Residual U-Net (FE + Residual blocks)
print("=" * 60)
print("  FINAL — ResidualUNet(14, 1)  |  12 bands + NDWI + MNDWI + Residual blocks")
print("=" * 60)

final_model = ResidualUNet(14, 1).to(device)
history_final, _ = train_model(
    final_model, train_loader_14, val_loader_14,
    name="final_residual", epochs=100, lr=1e-4, patience=10)


In [ ]:
#  Training curves 
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Training History", fontsize=13, fontweight="bold")

configs = [
    ("Baseline (12-ch UNet)",  history_baseline, "royalblue"),
    ("FE (14-ch UNet)",        history_fe,        "seagreen"),
    ("Final (14-ch Res-UNet)", history_final,     "tomato"),
]
for name, hist, color in configs:
    ep = range(1, len(hist["train_loss"]) + 1)
    axes[0].plot(ep, hist["train_loss"], label=name, color=color, linewidth=2)
    axes[1].plot(ep, hist["val_loss"],   label=name, color=color, linewidth=2)
    axes[2].plot(ep, hist["iou"],        label=name, color=color, linewidth=2)

for ax, title in zip(axes, ["Train Loss", "Val Loss", "Val IoU"]):
    ax.set_title(title); ax.set_xlabel("Epoch"); ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
#  Test-set evaluation for all 3 models 
criterion_eval = CombinedLoss(pos_weight=pos_weight.to(device))

results = {
    "Baseline (12-ch UNet)":   evaluate(baseline_model, test_loader_12, criterion_eval, device),
    "FE (14-ch UNet)":         evaluate(fe_model,       test_loader_14, criterion_eval, device),
    "Final (14-ch Res-UNet)":  evaluate(final_model,    test_loader_14, criterion_eval, device),
}

print("=" * 72)
print(f"  {'Model':<25} {'Loss':>7}  {'IoU':>7}  {'Prec':>7}  {'Rec':>7}  {'F1':>7}")
print("=" * 72)
for name, m in results.items():
    print(f"  {name:<25} {m['loss']:7.4f}  {m['iou']:7.4f}  "
          f"{m['precision']:7.4f}  {m['recall']:7.4f}  {m['f1']:7.4f}")
print("=" * 72)

# Bar chart comparison
metrics = ["iou", "precision", "recall", "f1"]
x       = range(len(metrics))
width   = 0.25
colors  = ["royalblue", "seagreen", "tomato"]
labels  = list(results.keys())

fig, ax = plt.subplots(figsize=(12, 5))
for i, (lbl, m, clr) in enumerate(zip(labels, results.values(), colors)):
    vals = [m[k] for k in metrics]
    bars = ax.bar([xi + i*width for xi in x], vals, width, label=lbl, color=clr, alpha=0.85)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks([xi + width for xi in x])
ax.set_xticklabels(["IoU", "Precision", "Recall", "F1"])
ax.set_ylim(0, 1.05)
ax.set_title("Test Set: Baseline vs Feature Engineering vs Final Model", fontsize=12, fontweight="bold")
ax.legend(); ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
